# Disaster Tweet Classification: Classical Machine Learning Benchmark with Joint Hyperparameter Tuning

**Model Pipelines (6 Models with Joint Feature & Classifier Hyperparameter Optimization):**
1. **BoW + LogisticRegression**
2. **BoW + LinearSVC**
3. **Word TF-IDF + LogisticRegression**
4. **Word TF-IDF + LinearSVC**
5. **BoW + MultinomialNB**
6. **Word TF-IDF + MultinomialNB**

---
### Comprehensive Hyperparameter Optimization Strategy:
* **Feature Vectorizer Grid (BoW & Word TF-IDF):**
  * `10k_unigram`: `ngram_range=(1, 1)`, `max_features=10,000`, `min_df=2`
  * `20k_bigram`: `ngram_range=(1, 2)`, `max_features=20,000`, `min_df=2`
  * `30k_trigram`: `ngram_range=(1, 3)`, `max_features=30,000`, `min_df=2`
  * `min_df2_all`: `ngram_range=(1, 2)`, `max_features=None`, `min_df=2` (All non-hapax tokens, ~114k)
* **Classifier Regularization Grid:**
  * Linear models (LR, LinearSVC): $C \in [0.1, 0.5, 1.0, 2.0, 5.0]$
  * Naive Bayes (MNB): $\alpha \in [0.01, 0.1, 0.5, 1.0]$
* **Artifacts Saved per Model:**
  * Full hyperparameter evaluation table (`hyperparam_tuning_results.csv`)
  * Validation curve plot across configurations (`hyperparam_tuning_plot.png`)
  * Serialized optimal model and vectorizer weights in `saved_models/`
  * Dual raw & normalized confusion matrix (`confusion_matrix.png`)
  * Per-class precision, recall, and F1 bar chart (`per_class_metrics.png`)
  * Classification reports (`classification_report.txt` and `.json`)
* **Benchmark Summary:** Master comparison table (`metrics_comparison.csv`) and vertical grouped bar chart (`models_metrics_comparison.png`).

In [ ]:
import os
import sys
import json
import time
import string
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Setup directories
DATA_DIR = Path("dataset")
RESULTS_DIR = Path("results/01_classical_ml_tfidf_bow")
MODELS_DIR = RESULTS_DIR / "saved_models"
PER_MODEL_DIR = RESULTS_DIR / "models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PER_MODEL_DIR, exist_ok=True)

train_path = DATA_DIR / "train_clean.parquet"
val_path = DATA_DIR / "validation_clean.parquet"
test_path = DATA_DIR / "test_clean.parquet"

# Sourcing dataset from Kaggle or Google Drive
KAGGLE_INPUT_DIR = Path("/kaggle/input/humaid-disaster-tweets-parquet")
if KAGGLE_INPUT_DIR.exists():
    print("[+] Sourcing dataset from Kaggle dataset input...")
    for split in ["train", "validation", "test"]:
        p_clean = KAGGLE_INPUT_DIR / f"{split}_clean.parquet"
        p_raw = KAGGLE_INPUT_DIR / f"{split}.parquet"
        target_p = DATA_DIR / f"{split}_clean.parquet"
        if not target_p.exists():
            if p_clean.exists():
                pd.read_parquet(p_clean).to_parquet(target_p)
            elif p_raw.exists():
                pd.read_parquet(p_raw).to_parquet(target_p)

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    raw_train = DATA_DIR / "train.parquet"
    raw_val = DATA_DIR / "validation.parquet"
    raw_test = DATA_DIR / "test.parquet"
    if not (raw_train.exists() and raw_val.exists() and raw_test.exists()):
        print("[+] Downloading HumAID dataset from Google Drive...")
        import gdown
        GDRIVE_URL = "https://drive.google.com/drive/folders/1pyMBc4SFc-sQvfmReiywPoN5cQbMpQBR?usp=drive_link"
        gdown.download_folder(url=GDRIVE_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)

train_file = train_path if train_path.exists() else DATA_DIR / "train.parquet"
val_file = val_path if val_path.exists() else DATA_DIR / "validation.parquet"
test_file = test_path if test_path.exists() else DATA_DIR / "test.parquet"

train_df = pd.read_parquet(train_file)
val_df = pd.read_parquet(val_file)
test_df = pd.read_parquet(test_file)

text_col = "clean_text" if "clean_text" in train_df.columns else "tweet_text"
target_col = "class_label"

print(f"[+] Loaded splits -> Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")

In [ ]:
# Light punctuation normalization
def preprocess_for_classical(text):
    if not isinstance(text, str):
        return ""
    translator = str.maketrans("", "", string.punctuation)
    return text.translate(translator).lower()

X_train_raw = train_df[text_col].apply(preprocess_for_classical).values
X_val_raw = val_df[text_col].apply(preprocess_for_classical).values
X_test_raw = test_df[text_col].apply(preprocess_for_classical).values

class_names = sorted(train_df[target_col].unique())
label2idx = {name: i for i, name in enumerate(class_names)}
idx2label = {i: name for i, name in enumerate(class_names)}

y_train = train_df[target_col].map(label2idx).values
y_val = val_df[target_col].map(label2idx).values
y_test = test_df[target_col].map(label2idx).values

display_name_map = {
    'caution_and_advice': 'Caution & Advice',
    'displaced_people_and_evacuations': 'Displaced / Evac.',
    'infrastructure_and_utility_damage': 'Infrastructure',
    'injured_or_dead_people': 'Injured / Dead',
    'missing_or_found_people': 'Missing / Found',
    'not_humanitarian': 'Not Humanitarian',
    'other_relevant_information': 'Other Info',
    'requests_or_urgent_needs': 'Requests / Urgent',
    'rescue_volunteering_or_donation_effort': 'Rescue / Donation',
    'sympathy_and_support': 'Sympathy / Support'
}
display_names = [display_name_map.get(c, c) for c in class_names]

print(f"[+] Encoded {len(class_names)} Target Classes:")
for i, name in enumerate(class_names):
    cnt = (y_train == i).sum()
    print(f"    Class {i:02d}: {display_name_map.get(name, name):<25} (Train Count: {cnt:,})")

In [ ]:
# Define vectorizer configurations
vec_configs = {
    "10k_unigram": {
        "bow": CountVectorizer(ngram_range=(1, 1), max_features=10000, min_df=2),
        "tfidf": TfidfVectorizer(ngram_range=(1, 1), max_features=10000, min_df=2, sublinear_tf=True)
    },
    "20k_bigram": {
        "bow": CountVectorizer(ngram_range=(1, 2), max_features=20000, min_df=2),
        "tfidf": TfidfVectorizer(ngram_range=(1, 2), max_features=20000, min_df=2, sublinear_tf=True)
    },
    "30k_trigram": {
        "bow": CountVectorizer(ngram_range=(1, 3), max_features=30000, min_df=2),
        "tfidf": TfidfVectorizer(ngram_range=(1, 3), max_features=30000, min_df=2, sublinear_tf=True)
    },
    "min_df2_all": {
        "bow": CountVectorizer(ngram_range=(1, 2), min_df=2),
        "tfidf": TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    }
}

# Pre-transform feature matrices for efficiency
print("[+] Fitting and transforming vectorizer feature matrices...")
feature_matrices = {}

for v_id, v_dict in vec_configs.items():
    print(f"    -> Building representations for config: '{v_id}'...")
    # BoW
    t0 = time.time()
    bow_v = v_dict["bow"]
    X_tr_bow = bow_v.fit_transform(X_train_raw)
    X_va_bow = bow_v.transform(X_val_raw)
    X_te_bow = bow_v.transform(X_test_raw)
    feature_matrices[(v_id, "bow")] = {
        "vectorizer": bow_v,
        "train": X_tr_bow,
        "val": X_va_bow,
        "test": X_te_bow,
        "dim": X_tr_bow.shape[1]
    }
    
    # TF-IDF
    tfidf_v = v_dict["tfidf"]
    X_tr_tfidf = tfidf_v.fit_transform(X_train_raw)
    X_va_tfidf = tfidf_v.transform(X_val_raw)
    X_te_tfidf = tfidf_v.transform(X_test_raw)
    feature_matrices[(v_id, "tfidf")] = {
        "vectorizer": tfidf_v,
        "train": X_tr_tfidf,
        "val": X_va_tfidf,
        "test": X_te_tfidf,
        "dim": X_tr_tfidf.shape[1]
    }
    print(f"       BoW dim: {X_tr_bow.shape[1]:,} | TF-IDF dim: {X_tr_tfidf.shape[1]:,} | Time: {time.time()-t0:.1f}s")

print("[+] All feature matrices built successfully.")

In [ ]:
def save_model_artifacts(model_name, best_clf, best_vec, y_true, y_pred, metrics_dict, tuning_df, param_name):
    model_slug = model_name.lower().replace(" ", "_").replace("+", "plus").replace("-", "_")
    curr_model_dir = PER_MODEL_DIR / model_slug
    os.makedirs(curr_model_dir, exist_ok=True)
    
    # 1. Hyperparameter tuning CSV & Sensitivity Plot
    tuning_df.to_csv(curr_model_dir / "hyperparam_tuning_results.csv", index=False)
    
    fig, ax = plt.subplots(figsize=(8.5, 5.5), dpi=300)
    for v_id in tuning_df['Vectorizer Config'].unique():
        sub_df = tuning_df[tuning_df['Vectorizer Config'] == v_id]
        ax.plot(sub_df[param_name], sub_df['Val Macro F1'], marker='o', linewidth=2, markersize=6, label=f"Vec: {v_id}")
        
    if param_name in ['C', 'alpha']:
        ax.set_xscale('log')
    ax.set_xlabel(f'Hyperparameter {param_name} (Log Scale)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Validation Macro F1 Score', fontsize=11, fontweight='bold')
    ax.set_title(f'Hyperparameter Tuning Curve: {model_name}', fontsize=12, fontweight='bold', pad=15)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='lower right', frameon=True, fontsize=9.5)
    plt.tight_layout()
    plt.savefig(curr_model_dir / "hyperparam_tuning_plot.png", bbox_inches='tight')
    plt.close()
    
    # 2. Classification Report (TXT and JSON)
    report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    with open(curr_model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(f"=== Classification Report: {model_name} (Optimized) ===\n\n")
        f.write(report_str)
        
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    with open(curr_model_dir / "classification_report.json", "w", encoding="utf-8") as f:
        json.dump(report_dict, f, indent=4)
        
    # 3. Metrics JSON
    with open(curr_model_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_dict, f, indent=4)
        
    # 4. Dual Confusion Matrix Plot (Raw and Normalized)
    cm_raw = confusion_matrix(y_true, y_pred)
    cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8.5), dpi=300)
    sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues',
                xticklabels=display_names, yticklabels=display_names,
                cbar=True, ax=ax1, annot_kws={"size": 8.5})
    ax1.set_title(f"Raw Confusion Matrix: {model_name}", fontsize=12, fontweight='bold', pad=12)
    ax1.set_xlabel("Predicted Class", fontsize=11, fontweight='bold', labelpad=8)
    ax1.set_ylabel("True Class", fontsize=11, fontweight='bold', labelpad=8)
    ax1.set_xticklabels(display_names, rotation=35, ha='right', fontsize=9.5)
    ax1.set_yticklabels(display_names, rotation=0, fontsize=9.5)
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=display_names, yticklabels=display_names,
                cbar=True, ax=ax2, annot_kws={"size": 8.5})
    ax2.set_title(f"Normalized Confusion Matrix: {model_name}", fontsize=12, fontweight='bold', pad=12)
    ax2.set_xlabel("Predicted Class", fontsize=11, fontweight='bold', labelpad=8)
    ax2.set_ylabel("True Class", fontsize=11, fontweight='bold', labelpad=8)
    ax2.set_xticklabels(display_names, rotation=35, ha='right', fontsize=9.5)
    ax2.set_yticklabels(display_names, rotation=0, fontsize=9.5)
    
    plt.tight_layout()
    plt.savefig(curr_model_dir / "confusion_matrix.png", bbox_inches='tight')
    plt.close()
    
    # 5. Per-Class Precision, Recall, and F1 Bar Chart
    per_class_df = pd.DataFrame([
        {
            "Class": display_name_map.get(cls, cls),
            "Precision": report_dict[cls]["precision"],
            "Recall": report_dict[cls]["recall"],
            "F1-Score": report_dict[cls]["f1-score"],
            "Support": report_dict[cls]["support"]
        }
        for cls in class_names
    ])
    per_class_df.to_csv(curr_model_dir / "per_class_metrics.csv", index=False)
    
    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
    x = np.arange(len(class_names))
    width = 0.25
    ax.barh(x - width, per_class_df["Precision"], width, label="Precision", color="#3498db")
    ax.barh(x, per_class_df["Recall"], width, label="Recall", color="#2ecc71")
    ax.barh(x + width, per_class_df["F1-Score"], width, label="F1-Score", color="#e74c3c")
    ax.set_yticks(x)
    ax.set_yticklabels(per_class_df["Class"], fontsize=9.5)
    ax.set_xlabel("Score", fontsize=10, fontweight='bold')
    ax.set_title(f"Per-Class Performance: {model_name}", fontsize=12, fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.savefig(curr_model_dir / "per_class_metrics.png", bbox_inches='tight')
    plt.close()

In [ ]:
# Define the 6 Model Pipelines and their search grids
model_specs = [
    {
        "name": "BoW + LogisticRegression",
        "feat_type": "bow",
        "feature_name": "Bag-of-Words",
        "clf_class": LogisticRegression,
        "param_name": "C",
        "param_grid": [0.1, 0.5, 1.0, 2.0, 5.0],
        "kwargs": {"class_weight": "balanced", "max_iter": 1000, "solver": "lbfgs", "random_state": 42}
    },
    {
        "name": "BoW + LinearSVC",
        "feat_type": "bow",
        "feature_name": "Bag-of-Words",
        "clf_class": LinearSVC,
        "param_name": "C",
        "param_grid": [0.05, 0.1, 0.5, 1.0, 2.0],
        "kwargs": {"class_weight": "balanced", "max_iter": 2000, "random_state": 42}
    },
    {
        "name": "Word TF-IDF + LogisticRegression",
        "feat_type": "tfidf",
        "feature_name": "Word TF-IDF",
        "clf_class": LogisticRegression,
        "param_name": "C",
        "param_grid": [0.1, 0.5, 1.0, 2.0, 5.0],
        "kwargs": {"class_weight": "balanced", "max_iter": 1000, "solver": "lbfgs", "random_state": 42}
    },
    {
        "name": "Word TF-IDF + LinearSVC",
        "feat_type": "tfidf",
        "feature_name": "Word TF-IDF",
        "clf_class": LinearSVC,
        "param_name": "C",
        "param_grid": [0.05, 0.1, 0.5, 1.0, 2.0],
        "kwargs": {"class_weight": "balanced", "max_iter": 2000, "random_state": 42}
    },
    {
        "name": "BoW + MultinomialNB",
        "feat_type": "bow",
        "feature_name": "Bag-of-Words",
        "clf_class": MultinomialNB,
        "param_name": "alpha",
        "param_grid": [0.01, 0.05, 0.1, 0.5, 1.0],
        "kwargs": {}
    },
    {
        "name": "Word TF-IDF + MultinomialNB",
        "feat_type": "tfidf",
        "feature_name": "Word TF-IDF",
        "clf_class": MultinomialNB,
        "param_name": "alpha",
        "param_grid": [0.01, 0.05, 0.1, 0.5, 1.0],
        "kwargs": {}
    },
]

master_benchmark_results = []

print("=== STARTING COMPREHENSIVE JOINT HYPERPARAMETER TUNING FOR 6 MODELS ===\n")

for spec in model_specs:
    name = spec["name"]
    feat_type = spec["feat_type"]
    clf_class = spec["clf_class"]
    param_name = spec["param_name"]
    param_grid = spec["param_grid"]
    base_kwargs = spec["kwargs"]
    
    print(f"[+] Optimizing Pipeline: {name}")
    tuning_records = []
    
    best_val_f1 = -1.0
    best_config = None
    best_param = None
    best_model = None
    best_vec = None
    best_test_mat = None
    
    # Sweep over all 4 vectorizer configurations
    for v_id in vec_configs.keys():
        f_data = feature_matrices[(v_id, feat_type)]
        X_tr = f_data["train"]
        X_va = f_data["val"]
        X_te = f_data["test"]
        vec_obj = f_data["vectorizer"]
        dim = f_data["dim"]
        
        # Sweep over classifier parameters
        for p_val in param_grid:
            t0 = time.time()
            clf_params = {param_name: p_val, **base_kwargs}
            clf = clf_class(**clf_params)
            clf.fit(X_tr, y_train)
            fit_time = time.time() - t0
            
            y_va_pred = clf.predict(X_va)
            val_acc = accuracy_score(y_val, y_va_pred)
            val_f1 = f1_score(y_val, y_va_pred, average="macro", zero_division=0)
            
            tuning_records.append({
                "Vectorizer Config": v_id,
                "Feature Dim": dim,
                param_name: p_val,
                "Val Macro F1": round(val_f1, 4),
                "Val Accuracy": round(val_acc, 4),
                "Train Time (s)": round(fit_time, 2)
            })
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_config = v_id
                best_param = p_val
                best_model = clf
                best_vec = vec_obj
                best_test_mat = X_te

    tuning_df = pd.DataFrame(tuning_records)
    print(f"    [*] Best Configuration: Vec='{best_config}' | {param_name}={best_param} (Val Macro F1: {best_val_f1:.4f})")
    
    # Evaluate optimal model on held-out test set
    y_test_pred = best_model.predict(best_test_mat)
    test_acc = float(accuracy_score(y_test, y_test_pred))
    test_macro_f1 = float(f1_score(y_test, y_test_pred, average='macro', zero_division=0))
    test_weighted_f1 = float(f1_score(y_test, y_test_pred, average='weighted', zero_division=0))
    test_prec_macro = float(precision_score(y_test, y_test_pred, average='macro', zero_division=0))
    test_rec_macro = float(recall_score(y_test, y_test_pred, average='macro', zero_division=0))
    
    metrics_dict = {
        "Model": name,
        "Feature Representation": spec["feature_name"],
        "Optimal Vectorizer": best_config,
        "Optimal Hyperparameter": f"{param_name}={best_param}",
        "Val Macro F1": round(best_val_f1, 4),
        "Test Accuracy": round(test_acc, 4),
        "Test Macro F1": round(test_macro_f1, 4),
        "Test Weighted F1": round(test_weighted_f1, 4),
        "Test Macro Precision": round(test_prec_macro, 4),
        "Test Macro Recall": round(test_rec_macro, 4)
    }
    
    # Save Model Weights & Vectorizer
    model_slug = name.lower().replace(" ", "_").replace("+", "plus").replace("-", "_")
    joblib.dump(best_model, MODELS_DIR / f"{model_slug}.joblib")
    joblib.dump(best_vec, MODELS_DIR / f"{model_slug}_vectorizer.joblib")
    
    # Save Artifacts
    save_model_artifacts(name, best_model, best_vec, y_test, y_test_pred, metrics_dict, tuning_df, param_name)
    
    master_benchmark_results.append(metrics_dict)
    print(f"    --> Held-Out Test Macro F1: {test_macro_f1:.4f} | Test Accuracy: {test_acc:.4f}\n")

In [ ]:
# Save Master Benchmark Comparison Table
results_df = pd.DataFrame(master_benchmark_results).sort_values(by="Test Macro F1", ascending=False).reset_index(drop=True)
results_df.to_csv(RESULTS_DIR / "metrics_comparison.csv", index=False)
print("=== FINAL 6-MODEL CLASSICAL BENCHMARK SUMMARY TABLE ===")
print(results_df.to_string(index=False))

# Plot Master Comparison (Vertical Grouped Bar Chart)
model_labels = []
for m in results_df["Model"]:
    m_label = m.replace("LogisticRegression", "LR").replace("LinearSVC", "LinearSVC").replace("MultinomialNB", "MNB")
    model_labels.append(m_label)

x = np.arange(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=1)

bars1 = ax.bar(x - width/2, results_df["Test Macro F1"], width, label="Macro F1", color="#1f77b4", edgecolor="none", zorder=3)
bars2 = ax.bar(x + width/2, results_df["Test Accuracy"], width, label="Accuracy", color="#ff7f0e", edgecolor="none", zorder=3)

for bar in bars1:
    h = bar.get_height()
    ax.annotate(f"{h:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9, fontweight='bold')
                
for bar in bars2:
    h = bar.get_height()
    ax.annotate(f"{h:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title("Classical Machine Learning Benchmark: Test Macro F1 vs Accuracy (Hyperparameter Tuned)", fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel("Model Pipeline", fontsize=11, fontweight='bold', labelpad=10)
ax.set_ylabel("Score", fontsize=11, fontweight='bold', labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=25, ha='right', fontsize=9.5)

max_val = max(results_df["Test Macro F1"].max(), results_df["Test Accuracy"].max())
ax.set_ylim(0, min(1.0, max_val + 0.12))
ax.legend(loc="upper left", frameon=True, fontsize=10)

for spine in ax.spines.values():
    spine.set_color('#888888')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "models_metrics_comparison.png", bbox_inches='tight')
plt.show()

print(f"\n[+] All 6 hyperparameter-tuned models, artifacts, and summary plots saved to: {RESULTS_DIR.resolve()}")